# Прогон spellcheck (BW/TEST_CLOUD_FINE_TUNE)

Инференс **[`BW/TEST_CLOUD_FINE_TUNE`](https://huggingface.co/BW/TEST_CLOUD_FINE_TUNE)** на тестовой выборке [`BW/RU_SPELLCHECK_DEVICE`](https://huggingface.co/datasets/BW/RU_SPELLCHECK_DEVICE) в трёх режимах:
- **seed prompt** — короткий system-промпт из [`gepa_spellcheck_prompt.json`](../gepa_spellcheck_prompt.json);
- **с промптом** — system-промпт из [`prompt.txt`](../prompt.txt);
- **Alpaca** — формат Cloud.ru FT: `instruction` из [`BW/spellcheck_benchmark_alpaca`](https://huggingface.co/datasets/BW/spellcheck_benchmark_alpaca) + `### Input` / `### Response`.
- **in-domain eval** — Alpaca-инференс на test из `spellcheck_benchmark_alpaca` (§12).

Результаты сохраняются в `finetune_seed_prompt_eval_predictions.csv`, `finetune_prompt_eval_predictions.csv`, `finetune_alpaca_prompt_eval_predictions.csv` и `finetune_alpaca_benchmark_test_predictions.csv`. Оценка **ERRANT** / **RUSpellEval** — скриптом [`evaluate_prompt_errant.py`](../utils/evaluate_prompt_errant.py).

**Нужно:** GPU (Colab T4+, рекомендуется 4-bit quantization).

**Runtime → Run all**

## 1. Окружение

In [1]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


## 2. Установка зависимостей

In [2]:
%%capture
import os

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install "transformers>=5.9.0" accelerate bitsandbytes "datasets==4.3.0" "huggingface_hub>=0.34.0"
else:
    !pip install "transformers>=5.9.0" accelerate bitsandbytes "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer


## 3. Загрузка промпта

In [3]:
from pathlib import Path
import json

PROMPT_CANDIDATES = [
    Path("../prompt.txt"),
    Path("prompt.txt"),
    Path("/content/NLP_PROJECT/prompt.txt"),
    Path("/content/prompt.txt"),
]

PROMPT_PATH = next((p for p in PROMPT_CANDIDATES if p.is_file()), None)
if PROMPT_PATH is None:
    raise FileNotFoundError(
        "prompt.txt не найден. Загрузите файл или клонируйте репозиторий."
    )

SYSTEM_PROMPT = PROMPT_PATH.read_text(encoding="utf-8").strip()
print(f"Loaded prompt from: {PROMPT_PATH.resolve()}")
print(f"Prompt length: {len(SYSTEM_PROMPT)} chars")
print("\n--- Preview ---")
print(SYSTEM_PROMPT[:400] + ("..." if len(SYSTEM_PROMPT) > 400 else ""))

GEPA_PROMPT_CANDIDATES = [
    Path("../gepa_spellcheck_prompt.json"),
    Path("gepa_spellcheck_prompt.json"),
    Path("/content/NLP_PROJECT/gepa_spellcheck_prompt.json"),
    Path("/content/gepa_spellcheck_prompt.json"),
]

GEPA_PROMPT_PATH = next((p for p in GEPA_PROMPT_CANDIDATES if p.is_file()), None)
if GEPA_PROMPT_PATH is None:
    SEED_PROMPT = (
        "Ты корректор русского текста. Исправь орфографические, пунктуационные "
        "и регистровые ошибки в предложении пользователя. Верни только исправленный "
        "текст без пояснений и кавычек."
    )
else:
    SEED_PROMPT = json.loads(GEPA_PROMPT_PATH.read_text(encoding="utf-8"))["seed_prompt"]

print(f"\nSeed prompt source: {GEPA_PROMPT_PATH.resolve() if GEPA_PROMPT_PATH else 'built-in fallback'}")
print(f"Seed prompt length: {len(SEED_PROMPT)} chars")
print(SEED_PROMPT)

Loaded prompt from: /content/prompt.txt
Prompt length: 1759 chars

--- Preview ---
You are a Russian‑language proofreader.  
For every input you receive, perform **only** the following actions:

1. **Correct** all orthographic (spelling), punctuation, and register/case mistakes.  
   - Fix misspelled words, missing or extra letters, and wrong word forms.  
   - Use proper Russian punctuation: commas, periods, question/exclamation marks, em‑dashes (—) for dialogue, hyphens (‑) on...

Seed prompt source: built-in fallback
Seed prompt length: 175 chars
Ты корректор русского текста. Исправь орфографические, пунктуационные и регистровые ошибки в предложении пользователя. Верни только исправленный текст без пояснений и кавычек.


## 4. Тестовый датасет RU_SPELLCHECK_DEVICE

In [4]:
from datasets import load_dataset

DATASET_NAME = "BW/RU_SPELLCHECK_DEVICE"
test_df = load_dataset(DATASET_NAME, split="test").to_pandas().dropna()

sources = test_df["typed"].tolist()
corrections = test_df["original"].tolist()

print(f"Loaded {DATASET_NAME} (test): {len(test_df)} rows")
print(f"Columns: {list(test_df.columns)}")
print("\nExample:")
print(f"  typed:    {sources[0]}")
print(f"  original: {corrections[0]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/598 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/69.4k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/67.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/296 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/296 [00:00<?, ? examples/s]

Loaded BW/RU_SPELLCHECK_DEVICE (test): 296 rows
Columns: ['id', 'created_at', 'player_id', 'device_type', 'original', 'typed', '__index_level_0__']

Example:
  typed:    - Графини , - отвечал будочник.
  original: — Графини , — отвечал будочник.


## 5. Загрузка BW/TEST_CLOUD_FINE_TUNE


In [ ]:
from huggingface_hub import hf_hub_download
from torch.cuda import temperature
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "BW/TEST_QWEN"
MAX_SEQ_LENGTH = 2048

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
chat_template_path = hf_hub_download(MODEL_NAME, "chat_template.jinja")
tokenizer.chat_template = open(chat_template_path, encoding="utf-8").read()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
import re


def strip_thinking_tags(text: str) -> str:
    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    think_close = "</think>"
    if think_close in text:
        text = text.split(think_close, maxsplit=1)[-1]
    return text.strip()


def predict_model(
    sentences: list[str],
    system_prompt: str | None = SYSTEM_PROMPT,
) -> list[str]:
    predictions = []
    for sentence in sentences:
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": sentence})

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        input_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max(int(len(sentence) * 1.5), 32),
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                temperature=0.0,
            )
        pred = strip_thinking_tags(
            tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
        )
        predictions.append(pred)
    return predictions


## 6. Инференс с seed prompt

In [15]:
demo_sentence = "И не чсно прохожим в этот день непогожйи почему я веселый такйо"
print("Demo (seed prompt):")
print(f"  input:  {demo_sentence}")
print(f"  output: {predict_model([demo_sentence], system_prompt=SEED_PROMPT)[0]}")

pred_model_seed = predict_model(sources, system_prompt=SEED_PROMPT)
print(f"Predictions (seed prompt): {len(pred_model_seed)}")


Demo (seed prompt):
  input:  И не чсно прохожим в этот день непогожйи почему я веселый такйо
  output: И не чсно прохожим в этот день непогожий и почему я весёлый такой
Predictions (seed prompt): 296


## 7. Сохранение предсказаний с seed prompt

In [16]:
from datetime import datetime, timezone

import pandas as pd

results_df_seed = pd.DataFrame({
    "typed": sources,
    "original": corrections,
    "prediction": pred_model_seed,
    "exact_match": [p == c for p, c in zip(pred_model_seed, corrections)],
    "model": MODEL_NAME,
    "prompt_path": str(GEPA_PROMPT_PATH.resolve()) if GEPA_PROMPT_PATH else "seed_prompt",
    "dataset": DATASET_NAME,
    "split": "test",
    "created_at": datetime.now(timezone.utc).isoformat(),
})

print(
    f"Exact match (seed prompt): "
    f"{results_df_seed['exact_match'].mean():.2%} "
    f"({results_df_seed['exact_match'].sum()}/{len(results_df_seed)})"
)
print("\n--- Первые 5 примеров ---")
for i, row in results_df_seed.head(5).iterrows():
    print(f"\n[{i + 1}]")
    print(f"  typed:       {row['typed']}")
    print(f"  original:    {row['original']}")
    print(f"  prediction:  {row['prediction']}")
    print(f"  exact_match: {row['exact_match']}")


Exact match (seed prompt): 17.23% (51/296)

--- Первые 5 примеров ---

[1]
  typed:       - Графини , - отвечал будочник.
  original:    — Графини , — отвечал будочник.
  prediction:  Графини, - отвечал будочник.
  exact_match: False

[2]
  typed:       ялидавта Ивановна решилась отвкчать.
  original:    Лизавета Ивановна решилась отвечать.
  prediction:  ялидавта Ивановна решилась отвкчать.
  exact_match: False

[3]
  typed:       Он шел;
  original:    Он шел;
  prediction:  Он шёл;
  exact_match: False

[4]
  typed:       вмдел в небе несчетные хвезды, светившие его путь и какк лев выступал сильно и бодро, так что когда восходящее солнце озарило своими влажнозкрасными лучами только что расходившегося молодца, мужду Москвой и им легко уже тридцать пять верст. Через два дня он уже был дома, в своей избенке, к велимому изумлению солдатки, которую туда поселили.
  original:    видел в небе несчетные звезды, светившие его путь, и как лев выступал сильно и бодро, так что когда восходящее 

In [17]:
SEED_OUTPUT_CANDIDATES = [
    Path("../finetune_seed_prompt_eval_predictions.csv"),
    Path("finetune_seed_prompt_eval_predictions.csv"),
    Path("/content/NLP_PROJECT/finetune_seed_prompt_eval_predictions.csv"),
    Path("/content/finetune_seed_prompt_eval_predictions.csv"),
]

SEED_OUTPUT_PATH = next(
    (p for p in SEED_OUTPUT_CANDIDATES if p.parent.exists()),
    Path("finetune_seed_prompt_eval_predictions.csv"),
)

results_df_seed.to_csv(SEED_OUTPUT_PATH, index=False)
print(f"Saved: {SEED_OUTPUT_PATH.resolve()}")
print("\nДля оценки ERRANT локально:")
print(f"  python utils/evaluate_prompt_errant.py --input {SEED_OUTPUT_PATH.resolve()}")


Saved: /finetune_seed_prompt_eval_predictions.csv

Для оценки ERRANT локально:
  python utils/evaluate_prompt_errant.py --input /finetune_seed_prompt_eval_predictions.csv


## 8. Инференс с промптом из prompt.txt

In [7]:
demo_sentence = "И не чсно прохожим в этот день непогожйи почему я веселый такйо"
print("Demo (с промптом из prompt.txt):")
print(f"  input:  {demo_sentence}")
print(f"  output: {predict_model([demo_sentence], system_prompt=SYSTEM_PROMPT)[0]}")

pred_model = predict_model(sources, system_prompt=SYSTEM_PROMPT)
print(f"Predictions (prompt.txt): {len(pred_model)}")

Demo (с промптом из prompt.txt):
  input:  И не чсно прохожим в этот день непогожйи почему я веселый такйо


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  output: И нечего прохожим в этот день непогоды почему я веселый такой
Predictions (prompt.txt): 296


## 9. Сохранение предсказаний с промптом

In [8]:
from datetime import datetime, timezone

import pandas as pd

results_df = pd.DataFrame({
    "typed": sources,
    "original": corrections,
    "prediction": pred_model,
    "exact_match": [p == c for p, c in zip(pred_model, corrections)],
    "model": MODEL_NAME,
    "prompt_path": str(PROMPT_PATH.resolve()),
    "dataset": DATASET_NAME,
    "split": "test",
    "created_at": datetime.now(timezone.utc).isoformat(),
})

print(f"Exact match: {results_df['exact_match'].mean():.2%} ({results_df['exact_match'].sum()}/{len(results_df)})")
print("\n--- Первые 5 примеров ---")
for i, row in results_df.head(5).iterrows():
    print(f"\n[{i + 1}]")
    print(f"  typed:       {row['typed']}")
    print(f"  original:    {row['original']}")
    print(f"  prediction:  {row['prediction']}")
    print(f"  exact_match: {row['exact_match']}")

Exact match: 32.43% (96/296)

--- Первые 5 примеров ---

[1]
  typed:       - Графини , - отвечал будочник.
  original:    — Графини , — отвечал будочник.
  prediction:  — Графинь , — отвечал будочник.
  exact_match: False

[2]
  typed:       ялидавта Ивановна решилась отвкчать.
  original:    Лизавета Ивановна решилась отвечать.
  prediction:  Лизавета Ивановна решилась отвечать.
  exact_match: True

[3]
  typed:       Он шел;
  original:    Он шел;
  prediction:  Он шел;
  exact_match: True

[4]
  typed:       вмдел в небе несчетные хвезды, светившие его путь и какк лев выступал сильно и бодро, так что когда восходящее солнце озарило своими влажнозкрасными лучами только что расходившегося молодца, мужду Москвой и им легко уже тридцать пять верст. Через два дня он уже был дома, в своей избенке, к велимому изумлению солдатки, которую туда поселили.
  original:    видел в небе несчетные звезды, светившие его путь, и как лев выступал сильно и бодро, так что когда восходящее солнце озарил

In [9]:
OUTPUT_CANDIDATES = [
    Path("../finetune_prompt_eval_predictions.csv"),
    Path("finetune_prompt_eval_predictions.csv"),
    Path("/content/NLP_PROJECT/finetune_prompt_eval_predictions.csv"),
    Path("/content/finetune_prompt_eval_predictions.csv"),
]

OUTPUT_PATH = next(
    (p for p in OUTPUT_CANDIDATES if p.parent.exists()),
    Path("finetune_prompt_eval_predictions.csv"),
)

results_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH.resolve()}")
print("\nДля оценки ERRANT локально:")
print(f"  python utils/evaluate_prompt_errant.py --input {OUTPUT_PATH.resolve()}")


Saved: /finetune_prompt_eval_predictions.csv

Для оценки ERRANT локально:
  python utils/evaluate_prompt_errant.py --input /finetune_prompt_eval_predictions.csv


In [10]:
!cp /finetune_prompt_eval_predictions.csv /content/finetune_prompt_eval_predictions.csv 
!cp /finetune_seed_prompt_eval_predictions.csv /content/finetune_seed_prompt_eval_predictions.csv 

cp: cannot stat '/finetune_seed_prompt_eval_predictions.csv': No such file or directory


## 10. Инференс в формате Alpaca (Cloud.ru FT)

Тот же `instruction`, что в [`BW/spellcheck_benchmark_alpaca`](https://huggingface.co/datasets/BW/spellcheck_benchmark_alpaca), и стандартный Alpaca-шаблон вместо chat template.

In [ ]:
from datasets import load_dataset

ALPACA_DATASET_NAME = "BW/spellcheck_benchmark_alpaca"

alpaca_row = load_dataset(ALPACA_DATASET_NAME, split="train")[0]
ALPACA_INSTRUCTION = alpaca_row["instruction"]

if ALPACA_INSTRUCTION.strip() != SYSTEM_PROMPT.strip():
    print("Warning: instruction from dataset differs from prompt.txt")
    print(f"  dataset instruction: {len(ALPACA_INSTRUCTION)} chars")
    print(f"  prompt.txt:          {len(SYSTEM_PROMPT)} chars")
else:
    print("Alpaca instruction matches prompt.txt")

ALPACA_PROMPT_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input that "
    "provides further context. Write a response that appropriately completes "
    "the request.\n\n"
    "### Instruction:\n"
    "{instruction}\n\n"
    "### Input:\n"
    "{input}\n\n"
    "### Response:\n"
)


def build_alpaca_prompt(instruction: str, input_text: str) -> str:
    return ALPACA_PROMPT_TEMPLATE.format(
        instruction=instruction,
        input=input_text,
    )


def predict_alpaca(
    sentences: list[str],
    instruction: str = ALPACA_INSTRUCTION,
) -> list[str]:
    predictions = []
    for sentence in sentences:
        prompt = build_alpaca_prompt(instruction, sentence)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        input_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max(int(len(sentence) * 1.5), 32),
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                temperature=0.0,
            )
        pred = strip_thinking_tags(
            tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
        )
        predictions.append(pred)
    return predictions


demo_sentence = "И не чсно прохожим в этот день непогожйи почему я веселый такйо"
print("Demo (Alpaca format):")
print(f"  input:  {demo_sentence}")
print(f"  output: {predict_alpaca([demo_sentence])[0]}")

Alpaca instruction matches prompt.txt
Demo (Alpaca format):
  input:  И не чсно прохожим в этот день непогожйи почему я веселый такйо


NameError: name 'model' is not defined

In [ ]:
pred_alpaca = predict_alpaca(sources)
print(f"Predictions (Alpaca): {len(pred_alpaca)}")

Predictions (Alpaca): 296


In [ ]:
a =2

## 11. Сохранение Alpaca-предсказаний

In [ ]:
from datetime import datetime, timezone

import pandas as pd

results_df_alpaca = pd.DataFrame({
    "typed": sources,
    "original": corrections,
    "prediction": pred_alpaca,
    "exact_match": [p == c for p, c in zip(pred_alpaca, corrections)],
    "model": MODEL_NAME,
    "prompt_path": f"alpaca:{ALPACA_DATASET_NAME}",
    "dataset": DATASET_NAME,
    "split": "test",
    "created_at": datetime.now(timezone.utc).isoformat(),
})

print(
    f"Exact match (Alpaca): "
    f"{results_df_alpaca['exact_match'].mean():.2%} "
    f"({results_df_alpaca['exact_match'].sum()}/{len(results_df_alpaca)})"
)
print("\n--- Первые 5 примеров ---")
for i, row in results_df_alpaca.head(5).iterrows():
    print(f"\n[{i + 1}]")
    print(f"  typed:       {row['typed']}")
    print(f"  original:    {row['original']}")
    print(f"  prediction:  {row['prediction']}")
    print(f"  exact_match: {row['exact_match']}")

Exact match (Alpaca): 28.04% (83/296)

--- Первые 5 примеров ---

[1]
  typed:       - Графини , - отвечал будочник.
  original:    — Графини , — отвечал будочник.
  prediction:  — Графини , — отвечал будочник.
  exact_match: True

[2]
  typed:       ялидавта Ивановна решилась отвкчать.
  original:    Лизавета Ивановна решилась отвечать.
  prediction:  Лизавета Ивановна решилась отвкнуть.
  exact_match: False

[3]
  typed:       Он шел;
  original:    Он шел;
  prediction:  Он шел;
  exact_match: True

[4]
  typed:       вмдел в небе несчетные хвезды, светившие его путь и какк лев выступал сильно и бодро, так что когда восходящее солнце озарило своими влажнозкрасными лучами только что расходившегося молодца, мужду Москвой и им легко уже тридцать пять верст. Через два дня он уже был дома, в своей избенке, к велимому изумлению солдатки, которую туда поселили.
  original:    видел в небе несчетные звезды, светившие его путь, и как лев выступал сильно и бодро, так что когда восходящее солн

In [ ]:
from pathlib import Path

ALPACA_OUTPUT_CANDIDATES = [
    Path("../finetune_alpaca_prompt_eval_predictions.csv"),
    Path("finetune_alpaca_prompt_eval_predictions.csv"),
    Path("/content/NLP_PROJECT/finetune_alpaca_prompt_eval_predictions.csv"),
    Path("/content/finetune_alpaca_prompt_eval_predictions.csv"),
]

ALPACA_OUTPUT_PATH = next(
    (p for p in ALPACA_OUTPUT_CANDIDATES if p.parent.exists()),
    Path("finetune_alpaca_prompt_eval_predictions.csv"),
)

results_df_alpaca.to_csv(ALPACA_OUTPUT_PATH, index=False)
print(f"Saved: {ALPACA_OUTPUT_PATH.resolve()}")
print("\nДля оценки ERRANT локально:")
print(f"  python utils/evaluate_prompt_errant.py --input {ALPACA_OUTPUT_PATH.resolve()}")

Saved: /finetune_alpaca_prompt_eval_predictions.csv

Для оценки ERRANT локально:
  python utils/evaluate_prompt_errant.py --input /finetune_alpaca_prompt_eval_predictions.csv


In [ ]:
!cp /finetune_alpaca_prompt_eval_predictions.csv /content/finetune_alpaca_prompt_eval_predictions.csv 

## 12. In-domain eval (`spellcheck_benchmark_alpaca` test)

Прогон **Alpaca**-инференса на test из [`BW/spellcheck_benchmark_alpaca`](https://huggingface.co/datasets/BW/spellcheck_benchmark_alpaca) — in-domain оценка относительно обучающего датасета.

По умолчанию берётся подвыборка (`IN_DOMAIN_MAX_SAMPLES`); для полного test (~12.1k) установите `None`.

In [11]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

IN_DOMAIN_MAX_SAMPLES = 500  # None — весь test (~12.1k)

benchmark_test = load_dataset(ALPACA_DATASET_NAME, split="test")
if IN_DOMAIN_MAX_SAMPLES is not None:
    benchmark_test = benchmark_test.select(
        range(min(IN_DOMAIN_MAX_SAMPLES, len(benchmark_test)))
    )

benchmark_sources = benchmark_test["input"]
benchmark_corrections = benchmark_test["output"]

print(f"Loaded {ALPACA_DATASET_NAME} (test): {len(benchmark_sources)} rows")

pred_benchmark = []
for sentence in tqdm(benchmark_sources, desc="In-domain Alpaca inference"):
    pred_benchmark.append(predict_alpaca([sentence])[0])

print(f"Predictions: {len(pred_benchmark)}")

NameError: name 'ALPACA_DATASET_NAME' is not defined

In [12]:
results_df_benchmark = pd.DataFrame({
    "typed": benchmark_sources,
    "original": benchmark_corrections,
    "prediction": pred_benchmark,
    "exact_match": [p == c for p, c in zip(pred_benchmark, benchmark_corrections)],
    "model": MODEL_NAME,
    "prompt_path": f"alpaca:{ALPACA_DATASET_NAME}",
    "dataset": ALPACA_DATASET_NAME,
    "split": "test",
    "created_at": datetime.now(timezone.utc).isoformat(),
})

print(
    f"Exact match (in-domain): "
    f"{results_df_benchmark['exact_match'].mean():.2%} "
    f"({results_df_benchmark['exact_match'].sum()}/{len(results_df_benchmark)})"
)
print("\n--- Первые 5 примеров ---")
for i, row in results_df_benchmark.head(5).iterrows():
    print(f"\n[{i + 1}]")
    print(f"  typed:       {row['typed'][:120]}{'...' if len(row['typed']) > 120 else ''}")
    print(f"  original:    {row['original'][:120]}{'...' if len(row['original']) > 120 else ''}")
    print(f"  prediction:  {row['prediction'][:120]}{'...' if len(row['prediction']) > 120 else ''}")
    print(f"  exact_match: {row['exact_match']}")

BENCHMARK_OUTPUT_CANDIDATES = [
    Path("../finetune_alpaca_benchmark_test_predictions.csv"),
    Path("finetune_alpaca_benchmark_test_predictions.csv"),
    Path("/content/NLP_PROJECT/finetune_alpaca_benchmark_test_predictions.csv"),
    Path("/content/finetune_alpaca_benchmark_test_predictions.csv"),
]

BENCHMARK_OUTPUT_PATH = next(
    (p for p in BENCHMARK_OUTPUT_CANDIDATES if p.parent.exists()),
    Path("finetune_alpaca_benchmark_test_predictions.csv"),
)

results_df_benchmark.to_csv(BENCHMARK_OUTPUT_PATH, index=False)
print(f"\nSaved: {BENCHMARK_OUTPUT_PATH.resolve()}")
print("\nДля оценки ERRANT локально:")
print(f"  python utils/evaluate_prompt_errant.py --input {BENCHMARK_OUTPUT_PATH.resolve()}")

NameError: name 'benchmark_sources' is not defined

In [13]:
!cp /finetune_alpaca_benchmark_test_predictions.csv /content/finetune_alpaca_benchmark_test_predictions.csv 

cp: cannot stat '/finetune_alpaca_benchmark_test_predictions.csv': No such file or directory
